# 05 — MLF vs. McDonald RG Comparison

Side-by-side accuracy and timing comparison of the two methods.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('../scripts').resolve()))

In [ ]:
from generate_mock import generate_synthetic_catalog
from pointpv.mock.catalog import eta_to_velocity
import pointpv.likelihood.mlf as mlf
import pointpv.likelihood.rg as rg

catalog = generate_synthetic_catalog(n=200, seed=42)
u = eta_to_velocity(catalog['eta'], catalog['z_obs'])
positions = catalog['pos']

fsigma8_grid = np.linspace(0.2, 0.8, 20)

res_mlf = mlf.scan_fsigma8(u, catalog, fsigma8_values=fsigma8_grid, verbose=False)
res_rg  = rg.scan_fsigma8(u, catalog, positions, fsigma8_values=fsigma8_grid, verbose=False)

In [ ]:
from pointpv.benchmark.accuracy import compare_logL, print_accuracy_report

acc = compare_logL(fsigma8_grid, res_mlf['logL'], res_rg['logL'])
print_accuracy_report(acc)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 8), sharex=True)

logL_mlf = res_mlf['logL']
logL_rg = res_rg['logL']
fs8 = fsigma8_grid

ax1.plot(fs8, logL_mlf - logL_mlf.max(), label='MLF (Cholesky)')
ax1.plot(fs8, logL_rg - logL_rg.max(), '--', label='McDonald RG')
ax1.axhline(-0.5, color='gray', linestyle=':', linewidth=0.8)
ax1.set_ylabel(r'$\Delta \log L$')
ax1.legend()
ax1.set_title('Log-likelihood profiles')

ax2.plot(fs8, logL_rg - logL_mlf, color='green')
ax2.axhline(0, color='gray', linestyle=':')
ax2.set_xlabel(r'$f\sigma_8$')
ax2.set_ylabel(r'$\log L_{RG} - \log L_{MLF}$')
ax2.set_title('Difference (RG - MLF)')

fig.tight_layout()
plt.show()

print(f'\nMLF mean eval time: {res_mlf["time_per_eval"].mean()*1000:.1f} ms')
print(f'RG  mean eval time: {res_rg["time_per_eval"].mean()*1000:.1f} ms')